# 0. Cellule de rechargement

In [1]:
# from pyspark.sql import SparkSession
# from pyspark.ml.feature import StringIndexer, VectorAssembler
# from pyspark.ml.classification import RandomForestClassifier
# from pyspark.ml.classification import RandomForestClassificationModel
# import pandas as pd
#
# # Spark session
# spark = SparkSession.builder.appName("NetSentinel").getOrCreate()
#
# # Charger le parquet
# df_final = spark.read.parquet("../data/processed/df_final.parquet")
# train_df, test_df = df_final.randomSplit([0.8, 0.2], seed=42)
#
# # Recréer l'indexer pour avoir le mapping labels
# df_clean = spark.read.parquet("../data/processed/df_final.parquet")
# indexer = StringIndexer(inputCol="label", outputCol="label_index")
#
# # Charger le modèle sauvegardé
# model = RandomForestClassificationModel.load("../data/model_rf")
#
# # Recréer les prédictions
# predictions = model.transform(test_df)
#
# print("Tout rechargé")

# 1. Initialisation de la session Spark

In [2]:
# initialisation de la session Spark
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("NetSentinel - Batch Analysis").config("spark.driver.memory", "8g").getOrCreate()
print(spark.version)

3.5.5


26/03/21 22:01:49 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


# 2. Concaténation des différents .csv

In [3]:
# importation de tous les .csv
import os

data_path = "../data"
df = spark.read.csv(data_path, header=True, inferSchema=True)

print(f"Nombre de lignes : {df.count()}")
print(f"Nombre de colonnes : {len(df.columns)}")

df.printSchema()

Nombre de lignes : 2608785
Nombre de colonnes : 122
root
 |-- flow_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- src_ip: string (nullable = true)
 |-- src_port: string (nullable = true)
 |-- dst_ip: string (nullable = true)
 |-- dst_port: string (nullable = true)
 |-- protocol: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- packets_count: string (nullable = true)
 |-- fwd_packets_count: string (nullable = true)
 |-- bwd_packets_count: string (nullable = true)
 |-- total_payload_bytes: string (nullable = true)
 |-- fwd_total_payload_bytes: string (nullable = true)
 |-- bwd_total_payload_bytes: string (nullable = true)
 |-- payload_bytes_max: string (nullable = true)
 |-- payload_bytes_min: string (nullable = true)
 |-- payload_bytes_mean: string (nullable = true)
 |-- payload_bytes_std: string (nullable = true)
 |-- payload_bytes_variance: string (nullable = true)
 |-- fwd_payload_bytes_max: double (nullable = true)
 |-- fwd_payload_by

### Vérification du déséquilibre des classes

- Le dataset source BCCC-CIC-IDS-2017 contient 2.6M de connexions réseau réparties sur 15 classes, Après analyse de la distribution des labels, nous avons décidé de filtrer le dataset pour ne garder que les 4 classes les plus représentées, afin de limiter le déséquilibre entre classes tout en conservant les types d'attaques les plus courants en entreprise

**info** : Note : le dataset source complet (2GB) est conservé et chargé intégralement par Spark —
le filtrage intervient uniquement en aval pour la phase d'entraînement du modèle.

In [4]:
df.groupBy("label").count().orderBy("count", ascending=False).show(truncate=False)

+-----------------+-------+
|label            |count  |
+-----------------+-------+
|Benign           |1786239|
|DoS_Hulk         |349240 |
|NULL             |170733 |
|Port_Scan        |161323 |
|DDoS_LOIT        |95733  |
|FTP-Patator      |9531   |
|DoS_GoldenEye    |8364   |
|DoS_Slowhttptest |6860   |
|SSH-Patator      |5949   |
|Botnet_ARES      |5508   |
|DoS_Slowloris    |5177   |
|Web_Brute_Force  |2734   |
|Web_XSS          |1358   |
|Web_SQL_Injection|24     |
|Heartbleed       |12     |
+-----------------+-------+



### Application des transformations citées ci-dessus

In [5]:
from pyspark.sql import functions as F

# définition des 10 labels qu'on veut garder
labels_conserved = [
    "Benign", "DoS_Hulk", "Port_Scan", "DDoS_LOIT",
    "FTP-Patator", "DoS_GoldenEye", "DoS_Slowhttptest",
    "SSH-Patator", "Botnet_ARES", "DoS_Slowloris"
]

# pour éviter le déséquilibre des classes, nous limitons le trafic normal a 100.000 lignes
benign_df = df.filter(F.col("label") == "Benign").limit(50000)

# pour éviter que les attaques avec peu de lignes soient écrasées nous allons également réduire Dos_Hulk + Port_Scan
dos_hulk_df = df.filter(F.col("label") == "DoS_Hulk").limit(50000)
port_scan_df = df.filter(F.col("label") == "Port_Scan").limit(50000)

# nous prenons toutes les attaques sauf le benign
attacks_df = df.filter(
    (F.col("label") != "Benign") &
    (F.col("label") != "DoS_Hulk") &
    (F.col("label") != "Port_Scan") &
    (F.col("label").isin(labels_conserved))
)

# nous collons les 2 df ensemble, le premier qui est limité à 100.000 et l'autre qui contient les attaques uniquement
df_balanced = benign_df.union(dos_hulk_df).union(port_scan_df).union(attacks_df)

# vérification - affiche le nombre de lignes par label
df_balanced.groupBy("label").count().orderBy("count", ascending=False).show()

+----------------+-----+
|           label|count|
+----------------+-----+
|       DDoS_LOIT|95733|
|          Benign|50000|
|        DoS_Hulk|50000|
|       Port_Scan|50000|
|     FTP-Patator| 9531|
|   DoS_GoldenEye| 8364|
|DoS_Slowhttptest| 6860|
|     SSH-Patator| 5949|
|     Botnet_ARES| 5508|
|   DoS_Slowloris| 5177|
+----------------+-----+



# 3. Exploration & Nettoyage des données

### 3.1 suppression des colonnes inutiles pour le ML

In [6]:
# colonnes à supprimer car elles sont inutiles pour le machine learning
cols_to_drop = ["flow_id", "timestamp", "src_ip", "dst_ip"]

# on crée un nouveau df en supprimant les colonnes
df_clean = df_balanced.drop(*cols_to_drop)

print(f"Colonnes restantes : {len(df_clean.columns)}")

Colonnes restantes : 118


### 3.2 affichage des types de chaque colonne

In [7]:
for col, dtype in df_clean.dtypes:
    print(f"{col} : {dtype}")

src_port : string
dst_port : string
protocol : string
duration : string
packets_count : string
fwd_packets_count : string
bwd_packets_count : string
total_payload_bytes : string
fwd_total_payload_bytes : string
bwd_total_payload_bytes : string
payload_bytes_max : string
payload_bytes_min : string
payload_bytes_mean : string
payload_bytes_std : string
payload_bytes_variance : string
fwd_payload_bytes_max : double
fwd_payload_bytes_min : string
fwd_payload_bytes_mean : double
fwd_payload_bytes_std : double
fwd_payload_bytes_variance : double
bwd_payload_bytes_max : double
bwd_payload_bytes_min : double
bwd_payload_bytes_mean : double
bwd_payload_bytes_std : double
bwd_payload_bytes_variance : double
total_header_bytes : double
max_header_bytes : double
min_header_bytes : int
mean_header_bytes : double
std_header_bytes : double
fwd_total_header_bytes : int
fwd_max_header_bytes : int
fwd_min_header_bytes : int
fwd_mean_header_bytes : double
fwd_std_header_bytes : double
bwd_total_header_by

### 3.3 conversion des dtypes nécessaire pour le ML
- Nous allons convertir les types (string) en types (double)

In [8]:
# liste des colonnes à convertir
string_to_double = [
    "src_port", "dst_port", "protocol", "duration", "packets_count",
    "fwd_packets_count", "bwd_packets_count", "total_payload_bytes",
    "fwd_total_payload_bytes", "bwd_total_payload_bytes",
    "payload_bytes_max", "payload_bytes_min", "payload_bytes_mean",
    "payload_bytes_std", "payload_bytes_variance", "fwd_payload_bytes_min"
]

for col_name in string_to_double:
    df_clean = df_clean.withColumn(col_name, F.col(col_name).cast("double"))

for col, dtype in df_clean.dtypes:
    print(f"{col} : {dtype}")

src_port : double
dst_port : double
protocol : double
duration : double
packets_count : double
fwd_packets_count : double
bwd_packets_count : double
total_payload_bytes : double
fwd_total_payload_bytes : double
bwd_total_payload_bytes : double
payload_bytes_max : double
payload_bytes_min : double
payload_bytes_mean : double
payload_bytes_std : double
payload_bytes_variance : double
fwd_payload_bytes_max : double
fwd_payload_bytes_min : double
fwd_payload_bytes_mean : double
fwd_payload_bytes_std : double
fwd_payload_bytes_variance : double
bwd_payload_bytes_max : double
bwd_payload_bytes_min : double
bwd_payload_bytes_mean : double
bwd_payload_bytes_std : double
bwd_payload_bytes_variance : double
total_header_bytes : double
max_header_bytes : double
min_header_bytes : int
mean_header_bytes : double
std_header_bytes : double
fwd_total_header_bytes : int
fwd_max_header_bytes : int
fwd_min_header_bytes : int
fwd_mean_header_bytes : double
fwd_std_header_bytes : double
bwd_total_header_by

### 3.4 vérification des valeurs

In [9]:
# - F.col(c).isNull() → retourne True/False pour chaque ligne
# - .cast("int") → convertit True=1, False=0
# - F.sum() → additionne tous les 1 = nombre total de nulls
# - .alias(c) → renomme la colonne résultante avec le nom de la colonne originale

null_counts = df_clean.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_clean.columns
])

# affichage vertical pour plus de clarté
null_counts.show(vertical=True)

26/03/21 22:02:39 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


-RECORD 0----------------------------
 src_port                   | 0      
 dst_port                   | 0      
 protocol                   | 287122 
 duration                   | 0      
 packets_count              | 0      
 fwd_packets_count          | 0      
 bwd_packets_count          | 0      
 total_payload_bytes        | 0      
 fwd_total_payload_bytes    | 0      
 bwd_total_payload_bytes    | 0      
 payload_bytes_max          | 0      
 payload_bytes_min          | 0      
 payload_bytes_mean         | 0      
 payload_bytes_std          | 0      
 payload_bytes_variance     | 0      
 fwd_payload_bytes_max      | 0      
 fwd_payload_bytes_min      | 0      
 fwd_payload_bytes_mean     | 0      
 fwd_payload_bytes_std      | 0      
 fwd_payload_bytes_variance | 0      
 bwd_payload_bytes_max      | 0      
 bwd_payload_bytes_min      | 0      
 bwd_payload_bytes_mean     | 0      
 bwd_payload_bytes_std      | 0      
 bwd_payload_bytes_variance | 0      
 total_heade

**info** : nous décidons de supprimer la colonne 'protocol' parce qu'elle contenait des valeurs non numériques (TCP, UDP) qui sont devenues nulle après la conversion en double, presque l'entièreté de la colonne est vide, de plus le protocole est indirectement représenté par les flags TCP donc nous décidons de supprimer cette colonne

In [10]:
# suppression de la colonne 'protocol'
df_clean = df_clean.drop("protocol")

# vérification
print(f"Colonnes restantes : {len(df_clean.columns)}")

Colonnes restantes : 117


### 3.5 remplacer les valeurs infinies par null

- quand spark divise par 0 il génère des inf ou -inf et le ML ne sait pas gérer ça, nous remplaçons donc les valeurs inf et -inf par null

In [11]:
import math

# nous récupérons toutes les colonnes doubles
double_cols = [c for c, t in df_clean.dtypes if t == "double"]

# pour chaque colonne double, on remplace par inf, -inf par null
for c in double_cols:
    df_clean = df_clean.withColumn(c,
        F.when(F.col(c) == float("inf"), None)
        .when(F.col(c) == float("-inf"), None)
        .otherwise(F.col(c))
    )

print(f"Valeurs infinies remplacées par null OK !")

Valeurs infinies remplacées par null OK !


In [12]:
# maintenant que certaines valeurs ont été remplacées par NULL, nous allons les supprimer

df_clean = df_clean.dropna()

# vérification
print(f"Lignes restantes : {df_clean.count()}")
df_clean.groupBy("label").count().orderBy("count", ascending=False).show()

Lignes restantes : 287122


+----------------+-----+
|           label|count|
+----------------+-----+
|       DDoS_LOIT|95733|
|          Benign|50000|
|        DoS_Hulk|50000|
|       Port_Scan|50000|
|     FTP-Patator| 9531|
|   DoS_GoldenEye| 8364|
|DoS_Slowhttptest| 6860|
|     SSH-Patator| 5949|
|     Botnet_ARES| 5508|
|   DoS_Slowloris| 5177|
+----------------+-----+



---
# 4. MACHINE LEARNING

### 4.1 encodage du label, on va remplacer le nom des attaques par des chiffres style label = 0, DDOS = 1 etc

In [13]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

# utilisation de StringIndexer qui permet de transformer les labels texte en indice numériques
indexer = StringIndexer(inputCol="label", outputCol="label_index")
df_indexed = indexer.fit(df_clean).transform(df_clean)

# assembler toutes les features en un seul vecteur, parce que spark accepte uniquement 2 colonnes en entrée donc on met toutes les feat dans un vec
feature_cols = [c for c in df_indexed.columns if c not in ["label", "label_index"]]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_final = assembler.transform(df_indexed)

# split 80 / 20
train_df, test_df = df_final.randomSplit([0.8, 0.2], seed=42)

print(f"Train : {train_df.count()}")
print(f"Test : {test_df.count()}")

Train : 229655


Test : 57467


### 4.2 sauvegarde intermédiaire .parquet

In [14]:
# afin de respecter les bonnes pratiques (conseillées) nous allons sauvegarder nos étapes intermédiaires en parquet

df_final.write.parquet("../data/processed/df_final.parquet", mode="overwrite")
print("Sauvegarde Parquet OK !")

# pour reprendre à ce niveau, run : df_final = spark.read.parquet("../data/processed/df_final.parquet")

Sauvegarde Parquet OK !


In [15]:
df_final = spark.read.parquet("../data/processed/df_final.parquet")
train_df, test_df = df_final.randomSplit([0.8, 0.2], seed=42)

### 4.3 train du modèle (random forest)

In [16]:
from pyspark.ml.classification import RandomForestClassifier

random_f = RandomForestClassifier(
    numTrees=100,
    labelCol="label_index",
    featuresCol="features",
    seed=42
)

print(f"Entraînement du modèle...")
model = random_f.fit(train_df)
print(f"Modèle entrainé et prêt !")

Entraînement du modèle...


Modèle entrainé et prêt !


### 4.4 évaluation

In [17]:
# prédictions sur le test set
predictions = model.transform(test_df)

# évaluation -> on utilise la multi classe pour évaluer et non un 1 vs 1
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# accuracy
accuracy = MulticlassClassificationEvaluator(
    labelCol="label_index",
    predictionCol="prediction",
    metricName="accuracy"
).evaluate(predictions)

# précision
precisions = MulticlassClassificationEvaluator(
    labelCol="label_index",
    predictionCol="prediction",
    metricName="weightedPrecision"
).evaluate(predictions)

# recall
recall = MulticlassClassificationEvaluator(
    labelCol="label_index",
    predictionCol="prediction",
    metricName="weightedRecall"
).evaluate(predictions)

# f1_score
f_score = MulticlassClassificationEvaluator(
    labelCol="label_index",
    predictionCol="prediction",
    metricName="f1"
).evaluate(predictions)

print(f"Accuracy : {accuracy}")
print(f"Précision : {precisions}")
print(f"Recall : {recall}")
print(f"F1-Score : {f_score}")

26/03/21 22:06:30 WARN DAGScheduler: Broadcasting large task binary with size 1004.0 KiB
26/03/21 22:06:32 WARN DAGScheduler: Broadcasting large task binary with size 1004.0 KiB
26/03/21 22:06:34 WARN DAGScheduler: Broadcasting large task binary with size 1004.0 KiB
26/03/21 22:06:36 WARN DAGScheduler: Broadcasting large task binary with size 1004.0 KiB


Accuracy : 0.92950922163176
Précision : 0.9395089658214473
Recall : 0.92950922163176
F1-Score : 0.9223687216792507


In [18]:
import pandas as pd
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# matrice de confusion
conf_matrix = predictions.groupBy("label_index", "prediction").count().orderBy("label_index", "prediction")

# conversion en pandas pour afficher
conf_df = conf_matrix.toPandas()

# Récupérer le mapping index -> nom d'attaque
labels = ["Benign", "DDoS_LOIT", "DoS_Hulk", "Port_Scan",
          "FTP-Patator", "DoS_GoldenEye", "DoS_Slowhttptest",
          "SSH-Patator", "Botnet_ARES", "DoS_Slowloris"]
print("Mapping :")
for i, l in enumerate(labels):
    print(f"  {i} → {l}")

print("\nMatrice de confusion brute :")
print(conf_df)

26/03/21 22:06:38 WARN DAGScheduler: Broadcasting large task binary with size 1005.5 KiB
26/03/21 22:06:40 WARN DAGScheduler: Broadcasting large task binary with size 1013.2 KiB
26/03/21 22:06:40 WARN DAGScheduler: Broadcasting large task binary with size 1014.5 KiB


Mapping :
  0 → Benign
  1 → DDoS_LOIT
  2 → DoS_Hulk
  3 → Port_Scan
  4 → FTP-Patator
  5 → DoS_GoldenEye
  6 → DoS_Slowhttptest
  7 → SSH-Patator
  8 → Botnet_ARES
  9 → DoS_Slowloris

Matrice de confusion brute :
    label_index  prediction  count
0           0.0         0.0  19086
1           0.0         1.0      4
2           0.0         3.0      8
3           1.0         0.0      3
4           1.0         1.0   9448
5           1.0         2.0    462
6           1.0         4.0      1
7           1.0         5.0      1
8           1.0         6.0      2
9           1.0         8.0      2
10          2.0         1.0      8
11          2.0         2.0   9958
12          2.0         4.0      3
13          2.0         5.0     11
14          3.0         1.0     30
15          3.0         2.0     11
16          3.0         3.0  10090
17          3.0         5.0      6
18          4.0         1.0      1
19          4.0         2.0    615
20          4.0         4.0   1328
21          5

26/03/21 22:06:40 WARN DAGScheduler: Broadcasting large task binary with size 1011.6 KiB


### Rappel -> prochaine étape :
- faire une matrice de confusion
- regardez, les features les plus importantes (faire quelques graphiques)
- quelques plots (distributions des attaques, métriques visuelle)
- faire le vrai dashboard